In [ ]:
import os

folder = r"E:\Prithu\Sustain\sustain train"
files  = os.listdir(folder)
for f in files:
    print(f)


In [ ]:
import os
import json

# Paths
BASE_DIR = r"C:\Users\06877\Documents\Sustain Image\sustain train"
JSON_FILE = os.path.join(BASE_DIR, "sustain.json")
IMAGE_DIR = os.path.join(BASE_DIR, "images")

# Load COCO JSON
with open(JSON_FILE, "r", encoding="utf-8") as f:
    coco_data = json.load(f)

# Get a set of the actual files currently in the folder (the renamed ones)
existing_files = set(os.listdir(IMAGE_DIR))

# Update each image entry
for img in coco_data["images"]:
    old_name = img["file_name"]
    
    # Predict what the new name should be using your original logic
    expected_new_name = old_name.replace(" ", "_") \
                                .replace("ä", "a") \
                                .replace("ö", "o") \
                                .replace("ü", "u") \
                                .replace("ß", "ss") \
                                .replace(",", "")
    
    # Check if this predicted new name is in the folder
    if expected_new_name in existing_files:
        img["file_name"] = expected_new_name  # Update the JSON entry
    else:
        print(f"⚠️  Not found in folder: {expected_new_name} (Original: {old_name})")

# Save fixed JSON
fixed_json_file = os.path.join(BASE_DIR, "sustain_fixed.json")
with open(fixed_json_file, "w", encoding="utf-8") as f:
    json.dump(coco_data, f, indent=2)

print(f"✅ JSON fixed and saved to {fixed_json_file}")

In [ ]:
import cv2

img_path = r"E:\Prithu\Sustain\sustain train\images\SUS23_01_Kathode_Ref1_langs_1000x_HF_01.jpg"
img = cv2.imread(img_path)
print("Loaded:", img is not None)  # Should print True

In [ ]:
import os, json, cv2, torch, torch.nn as nn
import numpy as np
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from pycocotools.coco import COCO
from tqdm import tqdm
import albumentations as A

# ─────────────────────────────────────────────────────────────
# 1. YOUR PATHS — Only edit this block
# ─────────────────────────────────────────────────────────────
BASE_DIR       = r"C:\Users\06877\Documents\Sustain Image\sustain train"
JSON_FILENAME  = "sustain_fixed.json"          # ← replace with your exact filename

# Auto-derived paths (no need to change)
TRAIN_JSON     = os.path.join(BASE_DIR, JSON_FILENAME)
IMAGE_DIR      = os.path.join(BASE_DIR, "images")
AUG_IMG_DIR    = os.path.join(BASE_DIR, "augmented", "images")
AUG_JSON       = os.path.join(BASE_DIR, "augmented", "annotations.json")
CKPT_DIR       = os.path.join(BASE_DIR, "checkpoints")
SAM3_WEIGHTS   = os.path.join(BASE_DIR, "sam3.pt")   # place sam3.pt here

os.makedirs(AUG_IMG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2. TRAINING CONFIG
# ─────────────────────────────────────────────────────────────
CFG = {
    "epochs"              : 80,
    "batch_size"          : 4,       # reduce to 2 if CUDA out of memory
    "lr"                  : 1e-4,
    "lr_min"              : 1e-6,
    "weight_decay"        : 1e-4,
    "grad_accum"          : 4,       # effective batch = 4 × 4 = 16
    "img_size"            : 2752,
    "focal_weight"        : 0.2,
    "dice_weight"         : 0.8,
    "warmup_epochs"       : 5,
    "freeze_epochs"       : 20,
    "early_stop_patience" : 10,
    "val_every"           : 5,
    "augments_per_image"  : 50,     # 17 × 100 = 1700 total samples
    "min_area"            : 25,
    "min_visibility"      : 0.2,
    "jpeg_quality"        : 95,
    "use_amp"             : True,
    "seed"                : 42,
}
torch.manual_seed(CFG["seed"])

In [ ]:
# 3. COCO HELPERS
# ─────────────────────────────────────────────────────────────
def polygons_to_mask(segmentation, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    for poly in segmentation:
        pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
        cv2.fillPoly(mask, [pts], 1)
    return mask

def mask_to_polygons(binary_mask):
    contours, _ = cv2.findContours(
        binary_mask.astype(np.uint8),
        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 4:
            continue
        poly = cnt.flatten().tolist()
        if len(poly) >= 6:
            polygons.append(poly)
    return polygons

def mask_to_coco_bbox(binary_mask):
    rows = np.any(binary_mask, axis=1)
    cols = np.any(binary_mask, axis=0)
    if not rows.any():
        return None
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return [int(cmin), int(rmin),
            int(cmax - cmin + 1), int(rmax - rmin + 1)]


In [ ]:
import albumentations as A
import cv2

def build_sem_safe_transform(img_size):
    """
    SEM-safe augmentation for Li-ion battery particles.
    Preserves particle shape, adds mild noise, lighting and flips.
    """
    transform = A.Compose([
        # ───────── GEOMETRIC ─────────
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
            scale=(0.9, 1.1),
            rotate=(-30, 30),
            mode=cv2.BORDER_REFLECT_101,
            p=0.5
        ),
        A.RandomCrop(width=img_size, height=img_size, p=1.0),
        
        A.Resize(img_size, img_size),

        # ───────── PHOTOMETRIC ─────────
        A.RandomBrightnessContrast(
            brightness_limit=0.25,
            contrast_limit=0.25,
            p=0.7
        ),
        A.RandomGamma(
            gamma_limit=(80, 120),
            p=0.5
        ),
        A.HueSaturationValue(
            hue_shift_limit=5,
            sat_shift_limit=15,
            val_shift_limit=15,
            p=0.3
        ),

        # ───────── NOISE ─────────
        A.OneOf([
            A.GaussNoise(p=1.0),
            A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.05, 0.2), p=1.0),
        ], p=0.4),

        # ───────── BLUR ─────────
        A.OneOf([
            A.GaussianBlur(blur_limit=(3,5), p=1.0),
            A.MedianBlur(blur_limit=3, p=1.0),
        ], p=0.2),

    ],
    bbox_params=A.BboxParams(
        format="coco",
        label_fields=["category_ids", "ann_ids"],
        min_area=0,        # adjust if needed
        min_visibility=0
    ),
    # If using masks:
    # mask_params=A.MaskParams()
    )

    return transform

In [ ]:
# 5. AUGMENTATION LOOP
# ─────────────────────────────────────────────────────────────
def run_augmentation():
    coco = COCO(TRAIN_JSON)
    with open(TRAIN_JSON) as f:
        coco_data = json.load(f)

    out_json = {
        "info"       : coco_data.get("info", {"description": "Augmented LIB Dataset"}),
        "licenses"   : coco_data.get("licenses", []),
        "categories" : coco_data["categories"],
        "images"     : [],
        "annotations": []
    }

    new_img_id = 1
    new_ann_id = 1
    transform = build_sem_safe_transform(CFG["img_size"])

    print(f"\n{'─'*58}")
    print(f"  Dataset   : {BASE_DIR}")
    print(f"  Images    : {len(coco_data['images'])}")
    print(f"  Augments  : {CFG['augments_per_image']} per image")
    print(f"  Expected  : {len(coco_data['images']) * CFG['augments_per_image']} total")
    print(f"  Output    : {AUG_IMG_DIR}")
    print(f"{'─'*58}\n")

    for img_info in tqdm(coco_data["images"], desc="Augmenting"):
        img_path = os.path.join(IMAGE_DIR, img_info["file_name"])
        image    = cv2.imread(img_path)

        if image is None:
            print(f"  ⚠️  Not found, skipping: {img_path}")
            continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        H, W  = image.shape[:2]

        ann_ids = coco.getAnnIds(imgIds=img_info["id"])
        anns    = coco.loadAnns(ann_ids)

        instance_masks   = []
        coco_bboxes      = []
        category_ids     = []
        original_ann_ids = []

        for ann in anns:
            mask = polygons_to_mask(ann["segmentation"], H, W)
            instance_masks.append(mask)
            coco_bboxes.append(ann["bbox"])
            category_ids.append(ann["category_id"])
            original_ann_ids.append(ann["id"])

        stem = os.path.splitext(img_info["file_name"])[0]

        for aug_idx in range(CFG["augments_per_image"]):
            np.random.seed(CFG["seed"] + new_img_id)

            result = transform(
                image        = image,
                masks        = instance_masks,
                bboxes       = coco_bboxes,
                category_ids = category_ids,
                ann_ids      = original_ann_ids
            )

            aug_img     = result["image"]
            aug_masks   = result["masks"]
            aug_bboxes  = result["bboxes"]
            aug_cat_ids = result["category_ids"]
            aug_H, aug_W = aug_img.shape[:2]

            new_fname = f"{stem}_aug_{aug_idx:04d}.jpg"
            save_path = os.path.join(AUG_IMG_DIR, new_fname)
            cv2.imwrite(
                save_path,
                cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, CFG["jpeg_quality"]]
            )

            out_json["images"].append({
                "id"         : new_img_id,
                "file_name"  : new_fname,
                "height"     : aug_H,
                "width"      : aug_W,
                "source_file": img_info["file_name"]
            })

            for mask, bbox, cat_id in zip(aug_masks, aug_bboxes, aug_cat_ids):
                polygons   = mask_to_polygons(mask)
                tight_bbox = mask_to_coco_bbox(mask)
                if not polygons or tight_bbox is None:
                    continue
                area = int(np.sum(mask))
                if area < CFG["min_area"]:
                    continue
                out_json["annotations"].append({
                    "id"          : new_ann_id,
                    "image_id"    : new_img_id,
                    "category_id" : int(cat_id),
                    "segmentation": polygons,
                    "bbox"        : tight_bbox,
                    "area"        : area,
                    "iscrowd"     : 0
                })
                new_ann_id += 1

            new_img_id += 1

    with open(AUG_JSON, "w") as f:
        json.dump(out_json, f, indent=2)

    print(f"\n{'─'*58}")
    print(f"  ✅  Augmentation done!")
    print(f"  Images      : {len(out_json['images'])}")
    print(f"  Annotations : {len(out_json['annotations'])}")
    print(f"  Saved to    : {AUG_JSON}")
    print(f"{'─'*58}\n")


In [ ]:
import random

class LIBSegDataset(Dataset):
    def __init__(self, coco_json, img_dir, img_size=1024):
        self.coco     = COCO(coco_json)
        self.img_dir  = img_dir
        self.img_size = img_size
        
        # Only keep images that actually have annotations
        self.img_ids  = [
            img_id for img_id in self.coco.imgs.keys() 
            if len(self.coco.getAnnIds(imgIds=img_id)) > 0
        ]

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id   = self.img_ids[idx]
        img_info = self.coco.imgs[img_id]
        img_path = os.path.join(self.img_dir, img_info["file_name"])

        # 1. Load and normalize image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Calculate scale factors for resizing bounding boxes later
        orig_H, orig_W = image.shape[:2]
        scale_x = self.img_size / orig_W
        scale_y = self.img_size / orig_H

        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        
        mean  = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std   = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std

        # 2. Pick ONE random annotation (particle) from this image to prompt the model
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        chosen_ann = self.coco.loadAnns(random.choice(ann_ids))[0]

        # 3. Get the mask for this specific particle
        m = self.coco.annToMask(chosen_ann)
        mask = cv2.resize(m, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0) # Shape: [1, H, W]

        # 4. Get and scale the bounding box prompt [x_min, y_min, x_max, y_max]
        x, y, w, h = chosen_ann["bbox"]
        bbox = torch.tensor([
            x * scale_x, 
            y * scale_y, 
            (x + w) * scale_x, 
            (y + h) * scale_y
        ], dtype=torch.float32)

        return image, bbox, mask

In [ ]:
# ─────────────────────────────────────────────────────────────
# 7. LOSS FUNCTIONS
# ─────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce   = nn.functional.binary_cross_entropy_with_logits(
            pred, target, reduction='none'
        )
        p_t   = torch.exp(-bce)
        focal = self.alpha * (1 - p_t) ** self.gamma * bce
        return focal.mean()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred  = torch.sigmoid(pred)
        inter = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return (1 - (2 * inter + self.smooth) / (union + self.smooth)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, focal_w=0.2, dice_w=0.8):
        super().__init__()
        self.focal = FocalLoss()
        self.dice  = DiceLoss()
        self.fw    = focal_w
        self.dw    = dice_w

    def forward(self, pred, target):
        return self.fw * self.focal(pred, target) + \
               self.dw * self.dice(pred, target)


In [ ]:
# ─────────────────────────────────────────────────────────────
# 9. TRAINING LOOP
# ─────────────────────────────────────────────────────────────
def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'─'*58}")
    print(f"  SAM3 Fine-Tuning — LIB Microscopy")
    print(f"  Device       : {device}")
    print(f"  Epochs       : {CFG['epochs']}")
    print(f"  Batch size   : {CFG['batch_size']}  (eff: {CFG['batch_size']*CFG['grad_accum']})")
    print(f"  Phase 1      : Decoder only  → epoch 1–{CFG['freeze_epochs']}")
    print(f"  Phase 2      : Full finetune → epoch {CFG['freeze_epochs']+1}–{CFG['epochs']}")
    print(f"{'─'*58}\n")

    from ultralytics.models.sam import SAM3SemanticPredictor
    overrides = dict(task="segment", mode="train", model=SAM3_WEIGHTS)
    predictor = SAM3SemanticPredictor(overrides=overrides)
    model     = predictor.model.to(device)

    train_ds = LIBSegDataset(AUG_JSON,    AUG_IMG_DIR, CFG["img_size"])
    val_ds   = LIBSegDataset(TRAIN_JSON,  IMAGE_DIR,   CFG["img_size"])

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                              shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=1,
                              shuffle=False, num_workers=2)

    print(f"  Train: {len(train_ds)} samples  |  Val: {len(val_ds)} samples\n")

    criterion = CombinedLoss(CFG["focal_weight"], CFG["dice_weight"]).to(device)
    optimizer = optim.AdamW(model.parameters(),
                            lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scheduler = WarmupCosineScheduler(
        optimizer, CFG["warmup_epochs"], CFG["epochs"], CFG["lr_min"]
    )
    scaler = GradScaler(enabled=CFG["use_amp"])

    best_val_loss    = float("inf")
    early_stop_count = 0
    history          = {"train_loss": [], "val_loss": [], "lr": []}

    for epoch in range(CFG["epochs"]):

        # Phase 1 — freeze encoder
        if epoch < CFG["freeze_epochs"]:
            for name, param in model.named_parameters():
                param.requires_grad = "image_encoder" not in name
            if epoch == 0:
                print("  [Phase 1] Image encoder FROZEN\n")

        # Phase 2 — unfreeze all, lower encoder LR
        elif epoch == CFG["freeze_epochs"]:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW([
                {"params": [p for n, p in model.named_parameters()
                            if "image_encoder" in n],      "lr": CFG["lr"] * 0.1},
                {"params": [p for n, p in model.named_parameters()
                            if "image_encoder" not in n],  "lr": CFG["lr"]},
            ], weight_decay=CFG["weight_decay"])
            print(f"\n  [Phase 2] Full model UNFROZEN at epoch {epoch+1}\n")

        # ------------------- TRAIN -------------------
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()

        # Update: Unpack 3 items (images, bboxes, masks)
        for step, (images, bboxes, masks) in enumerate(
            tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{CFG['epochs']}", leave=False)
        ):
            images = images.to(device)
            bboxes = bboxes.to(device)
            masks  = masks.to(device)

            with autocast(enabled=CFG["use_amp"]):
                # Format prompts: Add a dimension so shape is [Batch, 1, 4]
                prompt_bboxes = bboxes.unsqueeze(1)
                
                # Forward pass: Provide both image and prompts
                outputs = model(images, bboxes=prompt_bboxes)
                
                # Robustly extract the mask tensor from the Ultralytics output
                if isinstance(outputs, (tuple, list)):
                    pred_masks = outputs[0]
                elif isinstance(outputs, dict):
                    pred_masks = outputs.get("masks", outputs.get("pred_masks", outputs))
                else:
                    pred_masks = outputs

                # SAM outputs shape: [Batch, Num_Prompts, Num_Outputs, H, W] (e.g., [B, 1, 3, 1024, 1024])
                # We slice to get the first mask for our single prompt -> [B, 1, H, W]
                if pred_masks.dim() == 5:
                    primary_masks = pred_masks[:, 0, 0, :, :].unsqueeze(1)
                elif pred_masks.dim() == 4:
                    primary_masks = pred_masks[:, 0, :, :].unsqueeze(1)
                else:
                    primary_masks = pred_masks

                loss = criterion(primary_masks, masks) / CFG["grad_accum"]

            scaler.scale(loss).backward()

            if (step + 1) % CFG["grad_accum"] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            train_loss += loss.item() * CFG["grad_accum"]

        train_loss /= len(train_loader)

        # ----------------- VALIDATE -----------------
        val_loss = 0.0
        if (epoch + 1) % CFG["val_every"] == 0:
            model.eval()
            with torch.no_grad():
                # Update: Unpack 3 items here too
                for images, bboxes, masks in val_loader:
                    images = images.to(device)
                    bboxes = bboxes.to(device)
                    masks  = masks.to(device)
                    
                    with autocast(enabled=CFG["use_amp"]):
                        prompt_bboxes = bboxes.unsqueeze(1)
                        outputs = model(images, bboxes=prompt_bboxes)
                        
                        if isinstance(outputs, (tuple, list)):
                            pred_masks = outputs[0]
                        elif isinstance(outputs, dict):
                            pred_masks = outputs.get("masks", outputs.get("pred_masks", outputs))
                        else:
                            pred_masks = outputs

                        if pred_masks.dim() == 5:
                            primary_masks = pred_masks[:, 0, 0, :, :].unsqueeze(1)
                        elif pred_masks.dim() == 4:
                            primary_masks = pred_masks[:, 0, :, :].unsqueeze(1)
                        else:
                            primary_masks = pred_masks

                        val_loss += criterion(primary_masks, masks).item()
            val_loss /= len(val_loader)

            if val_loss < best_val_loss:
                best_val_loss    = val_loss
                early_stop_count = 0
                ckpt_path = os.path.join(CKPT_DIR, "sam3_lib_best.pt")
                torch.save({
                    "epoch"      : epoch + 1,
                    "model_state": model.state_dict(),
                    "optimizer"  : optimizer.state_dict(),
                    "val_loss"   : best_val_loss,
                    "cfg"        : CFG,
                }, ckpt_path)
                print(f"  ✅  Best model saved — val_loss = {best_val_loss:.4f}")
            else:
                early_stop_count += 1
                if early_stop_count >= CFG["early_stop_patience"]:
                    print(f"\n  ⛔  Early stop at epoch {epoch+1}")
                    break

        current_lr = scheduler.step(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        print(
            f"  Epoch {epoch+1:03d}/{CFG['epochs']} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f if val_loss else 'N/A':>7} | "
            f"LR: {current_lr:.2e}"
        )

    with open(os.path.join(CKPT_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n  Best val loss : {best_val_loss:.4f}")
    print(f"  Checkpoint    : {CKPT_DIR}\\sam3_lib_best.pt\n")

In [ ]:
import torch
import cv2
from ultralytics.models.sam import SAM3SemanticPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the base Ultralytics SAM3 model architecture
overrides = dict(task="segment", mode="predict", model=r"C:\Users\06877\Documents\Sustain Image\sustain train\sam3.pt")
predictor = SAM3SemanticPredictor(overrides=overrides)
model = predictor.model.to(device)

# 2. Load your newly trained custom weights
checkpoint = torch.load(r"C:\Users\06877\Documents\Sustain Image\sustain train\checkpoints\sam3_lib_best.pt")
model.load_state_dict(checkpoint["model_state"])
model.eval()

# 3. Run inference with a prompt (e.g., a bounding box [x_min, y_min, x_max, y_max])
image = cv2.imread("your_original_image.jpg")
# ... preprocess image to tensor (like in your dataset class) ...
prompt_box = torch.tensor([[[50.0, 50.0, 200.0, 200.0]]]).to(device) # Example box

with torch.no_grad():
    predictions = model(image_tensor, bboxes=prompt_box)
    # Extract and visualize the predicted mask!

In [ ]:
import os, json, cv2, torch, random
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from pycocotools.coco import COCO
from tqdm.auto import tqdm
import albumentations as A

# ─────────────────────────────────────────────────────────────
# 1. PATHS & CONFIG
# ─────────────────────────────────────────────────────────────
BASE_DIR       = r"C:\Users\06877\Documents\Sustain Image\sustain train"
JSON_FILENAME  = "sustain_fixed.json" 

TRAIN_JSON     = os.path.join(BASE_DIR, JSON_FILENAME)
IMAGE_DIR      = os.path.join(BASE_DIR, "images")
AUG_IMG_DIR    = os.path.join(BASE_DIR, "augmented", "images")
AUG_JSON       = os.path.join(BASE_DIR, "augmented", "annotations.json")
CKPT_DIR       = os.path.join(BASE_DIR, "checkpoints")
SAM3_WEIGHTS   = os.path.join(BASE_DIR, "sam3.pt") 

os.makedirs(AUG_IMG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)

CFG = {
    "epochs"              : 80,
    "batch_size"          : 2,       # Kept at 2 to ensure no OOM errors
    "lr"                  : 1e-4,
    "lr_min"              : 1e-6,
    "weight_decay"        : 1e-4,
    "grad_accum"          : 4,       
    "img_size"            : 1024,    # SAM's native size. We will crop to this!
    "focal_weight"        : 0.2,
    "dice_weight"         : 0.8,
    "warmup_epochs"       : 5,
    "freeze_epochs"       : 20,
    "early_stop_patience" : 10,
    "val_every"           : 5,
    "augments_per_image"  : 50,     
    "min_area"            : 25,
    "jpeg_quality"        : 95,
    "use_amp"             : True,
    "seed"                : 42,
}
torch.manual_seed(CFG["seed"])

# ─────────────────────────────────────────────────────────────
# 2. UTILS & DATASET
# ─────────────────────────────────────────────────────────────
def polygons_to_mask(segmentation, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)
    for poly in segmentation:
        pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
        cv2.fillPoly(mask, [pts], 1)
    return mask

def mask_to_polygons(binary_mask):
    contours, _ = cv2.findContours(binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 4: continue
        poly = cnt.flatten().tolist()
        if len(poly) >= 6: polygons.append(poly)
    return polygons

def mask_to_coco_bbox(binary_mask):
    rows = np.any(binary_mask, axis=1)
    cols = np.any(binary_mask, axis=0)
    if not rows.any(): return None
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return [int(cmin), int(rmin), int(cmax - cmin + 1), int(rmax - rmin + 1)]

class LIBSegDataset(Dataset):
    def __init__(self, coco_json, img_dir, img_size=1024):
        self.coco     = COCO(coco_json)
        self.img_dir  = img_dir
        self.img_size = img_size
        self.img_ids  = [img_id for img_id in self.coco.imgs.keys() if len(self.coco.getAnnIds(imgIds=img_id)) > 0]

    def __len__(self): return len(self.img_ids)

    def __getitem__(self, idx):
        img_id   = self.img_ids[idx]
        img_info = self.coco.imgs[img_id]
        img_path = os.path.join(self.img_dir, img_info["file_name"])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        orig_H, orig_W = image.shape[:2]
        scale_x = self.img_size / orig_W
        scale_y = self.img_size / orig_H

        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        
        mean  = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std   = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        chosen_ann = self.coco.loadAnns(random.choice(ann_ids))[0]

        m = self.coco.annToMask(chosen_ann)
        mask = cv2.resize(m, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0) 

        x, y, w, h = chosen_ann["bbox"]
        bbox = torch.tensor([x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y], dtype=torch.float32)

        return image, bbox, mask

# ─────────────────────────────────────────────────────────────
# 3. AUGMENTATION ENGINE
# ─────────────────────────────────────────────────────────────
def build_sem_safe_transform(img_size):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)}, scale=(0.9, 1.1), rotate=(-30, 30), border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.RandomCrop(width=img_size, height=img_size, p=1.0), # Crops 1024x1024 out of 2752x2208!
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=15, p=0.3),
        A.GaussNoise(p=0.4),
        A.OneOf([A.GaussianBlur(blur_limit=(3,5), p=1.0), A.MedianBlur(blur_limit=3, p=1.0)], p=0.2),
    ]) # Notice we removed BboxParams! We calculate them manually below to prevent crashes.

def run_augmentation():
    coco = COCO(TRAIN_JSON)
    with open(TRAIN_JSON) as f: coco_data = json.load(f)

    out_json = {
        "info": coco_data.get("info", {"description": "Augmented LIB Dataset"}),
        "licenses": coco_data.get("licenses", []), "categories": coco_data["categories"],
        "images": [], "annotations": []
    }

    new_img_id, new_ann_id = 1, 1
    transform = build_sem_safe_transform(CFG["img_size"])

    for img_info in tqdm(coco_data["images"], desc="Augmenting"):
        img_path = os.path.join(IMAGE_DIR, img_info["file_name"])
        image = cv2.imread(img_path)
        if image is None: continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        H, W  = image.shape[:2]
        ann_ids = coco.getAnnIds(imgIds=img_info["id"])
        anns = coco.loadAnns(ann_ids)

        instance_masks, category_ids = [], []
        for ann in anns:
            instance_masks.append(polygons_to_mask(ann["segmentation"], H, W))
            category_ids.append(ann["category_id"])

        stem = os.path.splitext(img_info["file_name"])[0]

        for aug_idx in range(CFG["augments_per_image"]):
            np.random.seed(CFG["seed"] + new_img_id)
            
            # Transform ONLY images and masks to prevent bbox errors
            result = transform(image=image, masks=instance_masks)
            aug_img, aug_masks = result["image"], result["masks"]
            aug_H, aug_W = aug_img.shape[:2]

            new_fname = f"{stem}_aug_{aug_idx:04d}.jpg"
            cv2.imwrite(os.path.join(AUG_IMG_DIR, new_fname), cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, CFG["jpeg_quality"]])

            out_json["images"].append({"id": new_img_id, "file_name": new_fname, "height": aug_H, "width": aug_W, "source_file": img_info["file_name"]})

            for mask, cat_id in zip(aug_masks, category_ids):
                polygons = mask_to_polygons(mask)
                tight_bbox = mask_to_coco_bbox(mask)
                
                # If the crop removed this particle entirely, tight_bbox is None, so we skip it!
                if not polygons or tight_bbox is None: continue
                area = int(np.sum(mask))
                if area < CFG["min_area"]: continue
                
                out_json["annotations"].append({
                    "id": new_ann_id, "image_id": new_img_id, "category_id": int(cat_id),
                    "segmentation": polygons, "bbox": tight_bbox, "area": area, "iscrowd": 0
                })
                new_ann_id += 1
            new_img_id += 1

    with open(AUG_JSON, "w") as f: json.dump(out_json, f, indent=2)

# ─────────────────────────────────────────────────────────────
# 4. TRAINING ENGINE
# ─────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, pred, target):
        bce = nn.functional.binary_cross_entropy_with_logits(pred, target, reduction='none')
        p_t = torch.exp(-bce)
        return (self.alpha * (1 - p_t) ** self.gamma * bce).mean()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        inter = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return (1 - (2 * inter + self.smooth) / (union + self.smooth)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, focal_w=0.2, dice_w=0.8):
        super().__init__()
        self.focal, self.dice = FocalLoss(), DiceLoss()
        self.fw, self.dw = focal_w, dice_w
    def forward(self, pred, target):
        return self.fw * self.focal(pred, target) + self.dw * self.dice(pred, target)

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, lr_min):
        self.optimizer, self.warmup_epochs, self.total_epochs = optimizer, warmup_epochs, total_epochs
        self.lr_max, self.lr_min = optimizer.param_groups[0]["lr"], lr_min
    def step(self, epoch):
        if epoch < self.warmup_epochs: lr = self.lr_max * (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1 + np.cos(np.pi * progress))
        for pg in self.optimizer.param_groups: pg["lr"] = lr
        return lr

def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    from ultralytics.models.sam import SAM3SemanticPredictor
    predictor = SAM3SemanticPredictor(overrides=dict(task="segment", mode="train", model=SAM3_WEIGHTS))
    model = predictor.model.to(device)

    train_ds = LIBSegDataset(AUG_JSON, AUG_IMG_DIR, CFG["img_size"])
    val_ds   = LIBSegDataset(TRAIN_JSON, IMAGE_DIR, CFG["img_size"])
    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

    criterion = CombinedLoss(CFG["focal_weight"], CFG["dice_weight"]).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scheduler = WarmupCosineScheduler(optimizer, CFG["warmup_epochs"], CFG["epochs"], CFG["lr_min"])
    scaler = GradScaler(enabled=CFG["use_amp"])

    best_val_loss, early_stop_count = float("inf"), 0

    for epoch in range(CFG["epochs"]):
        if epoch < CFG["freeze_epochs"]:
            for name, param in model.named_parameters(): param.requires_grad = "image_encoder" not in name
        elif epoch == CFG["freeze_epochs"]:
            for param in model.parameters(): param.requires_grad = True
            optimizer = optim.AdamW([
                {"params": [p for n, p in model.named_parameters() if "image_encoder" in n], "lr": CFG["lr"] * 0.1},
                {"params": [p for n, p in model.named_parameters() if "image_encoder" not in n], "lr": CFG["lr"]},
            ], weight_decay=CFG["weight_decay"])

        model.train()
        train_loss = 0.0
        optimizer.zero_grad()

        for step, (images, bboxes, masks) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)):
            images, bboxes, masks = images.to(device), bboxes.to(device), masks.to(device)
            with autocast(enabled=CFG["use_amp"]):
                outputs = model(images, bboxes=bboxes.unsqueeze(1))
                pred_masks = outputs[0] if isinstance(outputs, (tuple, list)) else outputs.get("masks", outputs) if isinstance(outputs, dict) else outputs
                
                primary_masks = pred_masks[:, 0, 0, :, :].unsqueeze(1) if pred_masks.dim() == 5 else pred_masks[:, 0, :, :].unsqueeze(1) if pred_masks.dim() == 4 else pred_masks
                loss = criterion(primary_masks, masks) / CFG["grad_accum"]

            scaler.scale(loss).backward()
            if (step + 1) % CFG["grad_accum"] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            train_loss += loss.item() * CFG["grad_accum"]

        train_loss /= len(train_loader)

        # Validation
        val_loss = 0.0
        if (epoch + 1) % CFG["val_every"] == 0:
            model.eval()
            with torch.no_grad():
                for images, bboxes, masks in val_loader:
                    images, bboxes, masks = images.to(device), bboxes.to(device), masks.to(device)
                    with autocast(enabled=CFG["use_amp"]):
                        outputs = model(images, bboxes=bboxes.unsqueeze(1))
                        pred_masks = outputs[0] if isinstance(outputs, (tuple, list)) else outputs.get("masks", outputs) if isinstance(outputs, dict) else outputs
                        primary_masks = pred_masks[:, 0, 0, :, :].unsqueeze(1) if pred_masks.dim() == 5 else pred_masks[:, 0, :, :].unsqueeze(1) if pred_masks.dim() == 4 else pred_masks
                        val_loss += criterion(primary_masks, masks).item()
            val_loss /= len(val_loader)

            if val_loss < best_val_loss:
                best_val_loss, early_stop_count = val_loss, 0
                torch.save({"model_state": model.state_dict()}, os.path.join(CKPT_DIR, "sam3_lib_best.pt"))
            else:
                early_stop_count += 1
                if early_stop_count >= CFG["early_stop_patience"]: break

        scheduler.step(epoch)
        print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f if val_loss else 'N/A':>7}")

# ─────────────────────────────────────────────────────────────
# 5. EXECUTION 
# ─────────────────────────────────────────────────────────────
print("⚙️ STARTING STAGE 1: Data Augmentation...")
run_augmentation()
print("\n⚙️ STARTING STAGE 2: SAM 3 Training...")
run_training()